In [ ]:
%pip install numpy pandas scipy scikit-learn

In [18]:
import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from collections import defaultdict, Counter

In [19]:
# bước chuẩn bị dữ liệu
%run Prepare_data.ipynb

Note: you may need to restart the kernel to use updated packages.
<class 'pandas.DataFrame'>
RangeIndex: 5220 entries, 0 to 5219
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             5220 non-null   int64         
 1   title_x            5220 non-null   str           
 2   text               5220 non-null   str           
 3   images_x           5220 non-null   object        
 4   asin               5220 non-null   str           
 5   parent_asin        5220 non-null   str           
 6   user_id            5220 non-null   str           
 7   timestamp          5220 non-null   datetime64[ns]
 8   helpful_vote       5220 non-null   int64         
 9   verified_purchase  5220 non-null   bool          
 10  main_category      5220 non-null   str           
 11  title_y            5220 non-null   str           
 12  average_rating     5220 non-null   float64       
 13  rating_n

# # Xây dựng mô hình Collaborative Filtering với SVD

In [20]:
def precision_at_k(recommended, actual, k=10):
    if not recommended:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    hits = len(set(recommended_k) & actual_set)
    return hits / k

def recall_at_k(recommended, actual, k=10):
    if not actual:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    hits = len(set(recommended_k) & actual_set)
    return hits / len(actual_set)

def ndcg_at_k(recommended, actual, k=10):
    if not recommended:
        return 0.0
    recommended_k = recommended[:k]
    actual_set = set(actual)
    dcg = 0.0
    for i, item in enumerate(recommended_k):
        if item in actual_set:
            dcg += 1.0 / np.log2(i + 2)
    ideal_hits = min(len(actual_set), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

In [21]:
def build_user_ground_truth(test_df):
    """
    Xây dựng ground truth cho bài toán user-to-item.
    Parameters:
        test_df: DataFrame chứa 'user_id', 'parent_asin', 'liked'
    Returns:
        dict: user_id -> set of parent_asin (các item liked trong test)
    """
    user_gt = test_df[test_df['liked'] == 1].groupby('user_id')['parent_asin'].apply(set).to_dict()
    return user_gt

def build_item_ground_truth(train_df, top_m=20):
    """
    Xây dựng ground truth cho bài toán item-to-item dựa trên co-occurrence.
    Parameters:
        train_df: DataFrame chứa 'user_id', 'parent_asin', 'liked' (chỉ lấy liked=1)
        top_m: số lượng item liên quan tối đa cho mỗi item (dùng để giới hạn ground truth)
    Returns:
        dict: parent_asin -> set of parent_asin (các item liên quan)
    """
    # Chỉ xét các tương tác liked=1
    liked_df = train_df[train_df['liked'] == 1]
    # Nhóm các item theo user
    user_items = liked_df.groupby('user_id')['parent_asin'].apply(set).to_dict()
    
    # Đếm số lần hai item cùng xuất hiện
    cooccur = defaultdict(Counter)
    for items in user_items.values():
        items_list = list(items)
        for i in range(len(items_list)):
            for j in range(i+1, len(items_list)):
                a, b = items_list[i], items_list[j]
                cooccur[a][b] += 1
                cooccur[b][a] += 1
    
    # Với mỗi item, lấy top_m item có số lần cùng xuất hiện cao nhất
    item_gt = {}
    for item, counter in cooccur.items():
        # Lấy top_m item (có thể ít hơn nếu không đủ)
        top_items = [it for it, _ in counter.most_common(top_m)]
        item_gt[item] = set(top_items)
    
    return item_gt

In [22]:
def evaluate_user_to_item(train_df, test_df, recommend_func, k=10):
    """
    Đánh giá mô hình user-to-item.
    """
    # Sử dụng hàm ground truth đã định nghĩa
    user_actual = build_user_ground_truth(test_df)  # thay vì viết trực tiếp
    user_interacted = train_df.groupby('user_id')['parent_asin'].apply(set).to_dict()
    
    precisions = []
    recalls = []
    ndcgs = []
    
    for user, actual in user_actual.items():
        if not actual:
            continue
        interacted = user_interacted.get(user, set())
        recommended = recommend_func(user, interacted, k)
        recommended = [item for item in recommended if item not in interacted]
        precisions.append(precision_at_k(recommended, actual, k))
        recalls.append(recall_at_k(recommended, actual, k))
        ndcgs.append(ndcg_at_k(recommended, actual, k))
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'ndcg': np.mean(ndcgs)
    }

In [23]:
def evaluate_item_to_item(train_df, item_ground_truth, recommend_func_item, k=10):
    """
    Đánh giá mô hình item-to-item.
    
    Parameters:
        train_df: không dùng trực tiếp, nhưng giữ để đồng bộ
        item_ground_truth: dict {item: set of related items}
        recommend_func_item: hàm func(item_id, k) -> list
        k: số lượng gợi ý
    """
    precisions = []
    recalls = []
    ndcgs = []
    
    for item, actual in item_ground_truth.items():
        if not actual:
            continue
        recommended = recommend_func_item(item, k)
        p = precision_at_k(recommended, actual, k)
        r = recall_at_k(recommended, actual, k)
        n = ndcg_at_k(recommended, actual, k)
        precisions.append(p)
        recalls.append(r)
        ndcgs.append(n)
    
    return {
        'precision': np.mean(precisions),
        'recall': np.mean(recalls),
        'ndcg': np.mean(ndcgs)
    }

In [24]:
def build_svd_model(train_df, n_factors=50):
    """
    Huấn luyện mô hình SVD bằng scipy.sparse.linalg.svds.
    
    Parameters:
        train_df: DataFrame có các cột 'user_id', 'parent_asin', 'liked'
        n_factors: số chiều latent factor
    
    Returns:
        user_embeddings: dict {user_id: embedding vector}
        item_embeddings: dict {item_id: embedding vector}
        all_items: list of all item ids
    """
    # Tạo mapping user_id -> internal index
    users = train_df['user_id'].unique()
    items = train_df['parent_asin'].unique()
    user_to_idx = {user: i for i, user in enumerate(users)}
    item_to_idx = {item: i for i, item in enumerate(items)}
    idx_to_item = {i: item for item, i in item_to_idx.items()}
    
    # Tạo ma trận tương tác user-item (dense)
    n_users = len(users)
    n_items = len(items)
    interaction_matrix = np.zeros((n_users, n_items))
    
    for _, row in train_df.iterrows():
        user_idx = user_to_idx[row['user_id']]
        item_idx = item_to_idx[row['parent_asin']]
        interaction_matrix[user_idx, item_idx] = row['liked']
    
    # Chuẩn hóa dữ liệu (có thể dùng MinMaxScaler nếu cần)
    # interaction_matrix = MinMaxScaler().fit_transform(interaction_matrix)
    
    # Phân rã ma trận bằng SVD (chỉ lấy n_factors thành phần chính)
    # Giá trị k (số lượng singular values) không được vượt quá min(n_users, n_items)
    k = min(n_factors, min(n_users, n_items) - 1)
    if k < 1:
        k = 1
    
    U, sigma, Vt = svds(interaction_matrix, k=k)
    sigma = np.diag(sigma)
    
    # Tạo embedding cho user và item
    user_embeddings = U @ sigma
    item_embeddings = Vt.T
    
    # Chuyển về dạng dict
    user_emb_dict = {users[i]: user_embeddings[i] for i in range(n_users)}
    item_emb_dict = {items[i]: item_embeddings[i] for i in range(n_items)}
    
    return user_emb_dict, item_emb_dict, items

def create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df):
    """
    Tạo các hàm recommend cho user-to-item và item-to-item dựa trên SVD.
    """
    # --- Hàm recommend cho user-to-item ---
    def recommend_user(user_id, interacted_items, k=10):
        if user_id not in user_embeddings:
            return []
        user_vec = user_embeddings[user_id]
        # Dự đoán điểm cho tất cả item
        predictions = []
        for item in all_items:
            if item in interacted_items:
                continue
            if item not in item_embeddings:
                continue
            item_vec = item_embeddings[item]
            # Dùng dot product làm điểm dự đoán
            score = np.dot(user_vec, item_vec)
            predictions.append((item, score))
        predictions.sort(key=lambda x: x[1], reverse=True)
        return [item for item, _ in predictions[:k]]
    
    # --- Hàm recommend cho item-to-item (dùng cosine similarity) ---
    # Tạo ma trận embeddings cho tất cả items
    all_items_list = list(item_embeddings.keys())
    embedding_matrix = np.vstack([item_embeddings[item] for item in all_items_list])
    
    def recommend_item(item_id, k=10):
        if item_id not in item_embeddings:
            return []
        target_emb = item_embeddings[item_id].reshape(1, -1)
        sim_scores = cosine_similarity(target_emb, embedding_matrix)[0]
        top_indices = np.argsort(-sim_scores)[1:k+1]
        return [all_items_list[i] for i in top_indices]
    
    return recommend_user, recommend_item

# Điều chỉnh Hyperparameter cho tối ưu nhất

### đánh giá với n_factors = 50

In [26]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=50)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, valid_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0022
Recall@10: 0.0144
NDCG@10: 0.0079

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.3601
Recall@10: 0.3532
NDCG@10: 0.4805


### đánh giá với n_factors = 25

In [27]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=25)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, valid_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0020
Recall@10: 0.0131
NDCG@10: 0.0063

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.3143
Recall@10: 0.3043
NDCG@10: 0.4276


### đánh giớ với n_factor = 75

In [28]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=75)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, valid_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0020
Recall@10: 0.0116
NDCG@10: 0.0075

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.3918
Recall@10: 0.3889
NDCG@10: 0.5140


### đánh giá với n_factors = 90

In [30]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=90)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, valid_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0020
Recall@10: 0.0122
NDCG@10: 0.0072

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.4119
Recall@10: 0.4110
NDCG@10: 0.5340


### đánh giá với n_factor = 110

In [31]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=110)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, valid_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0023
Recall@10: 0.0153
NDCG@10: 0.0094

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.4385
Recall@10: 0.4433
NDCG@10: 0.5616


### đánh giá với n_factors = 120

In [35]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=120)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, valid_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0018
Recall@10: 0.0133
NDCG@10: 0.0076

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.4411
Recall@10: 0.4474
NDCG@10: 0.5635


### đánh giá với n_factors = 130

In [32]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=130)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, valid_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0018
Recall@10: 0.0140
NDCG@10: 0.0079

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.4523
Recall@10: 0.4611
NDCG@10: 0.5769


### đánh giá với n_factors = 150

In [34]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=150)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, valid_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0012
Recall@10: 0.0084
NDCG@10: 0.0066

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.4663
Recall@10: 0.4812
NDCG@10: 0.5970


# Kết quả đánh mô hình với n_factors tối ưu nhất ( n_factor = 110 )

In [43]:
# Huấn luyện mô hình SVD
user_embeddings, item_embeddings, all_items = build_svd_model(train_df, n_factors=110)

# Tạo các hàm recommend
recommend_user, recommend_item = create_svd_recommenders(user_embeddings, item_embeddings, all_items, train_df)

# Đánh giá user-to-item
user_results = evaluate_user_to_item(train_df, test_df, recommend_user, k=10)
print("=== Collaborative Filtering (SVD) - User-to-item ===")
print(f"Precision@10: {user_results['precision']:.4f}")
print(f"Recall@10: {user_results['recall']:.4f}")
print(f"NDCG@10: {user_results['ndcg']:.4f}")

# Đánh giá item-to-item (cần ground truth từ train)
item_gt = build_item_ground_truth(train_df, top_m=20)
item_results = evaluate_item_to_item(train_df, item_gt, recommend_item, k=10)
print("\n=== Collaborative Filtering (SVD) - Item-to-item ===")
print(f"Precision@10: {item_results['precision']:.4f}")
print(f"Recall@10: {item_results['recall']:.4f}")
print(f"NDCG@10: {item_results['ndcg']:.4f}")

=== Collaborative Filtering (SVD) - User-to-item ===
Precision@10: 0.0013
Recall@10: 0.0097
NDCG@10: 0.0057

=== Collaborative Filtering (SVD) - Item-to-item ===
Precision@10: 0.4367
Recall@10: 0.4410
NDCG@10: 0.5597
